# `TRG_EMP_LOAD.txt` Conversion to Databricks Spark SQL

**Converted On:** 2024-07-30

**Description:** This notebook loads data into the `TRG_EMP` table from the `EMPLOYEES` table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

The following parameters are used to control the ETL process and track lineage.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
    '${ETL_JOB_TYPE}' AS etl_job_type,
    '${DATASOURCE_NUM_ID}' AS datasource_num_id,
    '${ETL_PROC_WID}' AS etl_proc_wid,
    '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

# SCEN_TASK_NO {10}

# SCEN_TASK_NO {20}

# Load Data into Target Table

**SCEN_TASK_NO {30}:** Inserts all records from the source `EMPLOYEES` table into the target `TRG_EMP` table.

In [ ]:
%sql
INSERT INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id 
  ) 
SELECT 
  EMPLOYEES.employee_id ,
  EMPLOYEES.first_name ,
  EMPLOYEES.last_name ,
  EMPLOYEES.email ,
  EMPLOYEES.phone_number ,
  EMPLOYEES.hire_date ,
  EMPLOYEES.job_id ,
  EMPLOYEES.salary ,
  EMPLOYEES.commission_pct ,
  EMPLOYEES.manager_id ,
  EMPLOYEES.department_id  
FROM 
  workspace.hr.employees AS EMPLOYEES;

# Optimize Target Table

Optimizing the target table for better query performance.

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_emp ZORDER BY (employee_id, department_id);

# Validation

Count of records in the target table after the load.

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_emp;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_emp LIMIT 10;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: All schema names (`HR`) have been converted to `workspace.hr` and table names to lowercase (`TRG_EMP` to `trg_emp`, `EMPLOYEES` to `employees`).
2.  **Oracle Hints**: The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Delta Lake.
3.  **Data Types**: No `CREATE TABLE` DDL was provided in the source SQL, so explicit data type conversions (e.g., `VARCHAR2` to `STRING`, `NUMBER` to `BIGINT`/`DECIMAL`, `DATE` to `TIMESTAMP`) were not performed in this notebook. It is assumed the `TRG_EMP` table already exists with compatible Spark SQL data types.
4.  **Optimization**: An `OPTIMIZE ... ZORDER BY` statement was added to the target table for improved query performance, preceded by `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` as per best practices.
5.  **Parameter Handling**: ODI global/schema parameters (if any were present) would be converted to Databricks widgets. For this specific snippet, no parameters were identified in the `INSERT` statement itself. Generic ETL widgets have been included.